# Rate of Change Method

The rate of change method tracks the daily rate of change in chlorophyll-a concentration and finds the maximum rates of change throughout the time series. These maximum rates of change are designated as bloom initiation dates. This method does not rely on climatological values, it focuses on a daily differences and compares them to other values after a set period of days. You can set the number of days between blooms in order to eliminate extra peaks from small spikes in the bloom. Prior to the rate of change being calculated, the chlorophyll-a data should be smoothed so rates of change accurately represent blooms.

## Spatially averaged rates of change

#### Lowess smoothing of chlorophyll data
Lowess smoothing uses the points surrounding a point (the fraction of points surrounding a data point) to estimate the value of that point (statsmodels webpage). This helps smooth out the naturally noisy chlorophyll-a data. Since the data has been previously smoothed using a rolling 8-day mean, only light smoothing is necessary to remove the jagged nature of bloom peaks. For this reason, a fractional value of 0.035 was chosen. This value accurately smooths out neglible chlorophyll-a concentration spikes, while still accurately modelling the major chl-a events. For verfication of the fractional value chosen, a time series of one year of data was plotted alongside its smoothed value. This process was repeated with a variety of years in the time series and in all of the regions of the NES. A fraction value of 0.035 smooths over approximately 12.77 days in a year long time series. To keep this level of smoothing consistent, a fractional value of 0.0013 was chosen to smooth the entire time series. This provides the same smoothing resolution as the 0.035 smoother on the yearly data.

#### Calculating the rate of change over a small spatially averaged area

The rate_of_change function spatially averages the chlorophyll a data, smooths the data using a lowess smoother, and then find the rates of change between each data point, creating an array of rates of change.
* Must include:
    * lat_min
    * lat_max
    * lon_min
    * lon_max
    * path (set to weekly climatology by default)

The max_roc function builds on the rate_of_change function and finds the maximum rates of change in the array based on the parameters set in the function.
* Must include:
    * lat_min
    * lat_max
    * lon_min
    * lon_max
    * path (set to weekly climatology by default)
    * Distance = number of days between peaks (set to 10 days by default)
    * Prominence = the compared value of a rate of change to the next in order for it to be a peak (set to 0.02 by default)

For the find_peaks function inside, we have chosen 10 days as the distance necessary between peaks and 0.02 as the prominence. After testing values around 10 days, including 7, 14, and 21, I found that there were minor differences between 7, 10, and 14 days. Twenty-one days eliminated some peaks that were considered separate events based on climatological medians and timing. When testing prominence, there was very minor differences between values in the range 0.005 and 0.02. After 0.02, the function struggled to pick up any peaks due to the variable nature of chlorophyll-a satellite data. 

#### Regional Rates of Change

This involves spatially averaging the chlorophyll-a data by the regional chapefiles for each of the ECUs in the NES: Middle Atlantic South, Middle Atlantic North, Georges Bank, Gulf of Maine West, Gulf of Maine East.

The instant_rate_of_change function takes a smoothed dataset and calculates the gradient between every point. This function is used in other functions, not necessarily as a stand alone function. 
* Must include:
    * Dataset (this function smooths the data at a frac value of 0.04)

The max_roc_dates function finds the maximum rates of change and returns their date. It utilizes the find_peaks() function from scipy, with the prominence (prm) set at 0.015 by default and the distance (days) set as 10 by default. A prominence of 0.02 was chosen after testing a variety of values centered around 0.02. Anything higher than 0.02 risked cutting out the major peaks in some years and anything less than 0.01 included minor daily variations in chlorophyll concentration. Based on time series plots of the data, a prominence of 0.015 was able to capture separate bloom events without capturing a lot of noise. THe distance was set as 10 arbitrarily. There is not large difference in points detected when using values between 5 and 20.
* Must include:
    * Dataset (unsmoothed as this function uses instant_rate_of_change)
    * Prominence (prm) (set to 0.02 by default)
    * Distance (days) (set to 10 by default)

The bloom_peak_detection function finds all peaks in a dataset as long as the peaks rise above the 10% median climatological threshold set by the threshold method. 
* Must include:
    * Dataset
    * Index (0-4) for climatological median (MABS, MABN, GB, GOMW, GOME)
    * days (=10 by default)
    * prominence (prm=0.02 by default)



The bloom_event_detection function takes the peaks found in the bloom_peak_detection function and combines them into groups based on proximity. For this function, event_distance denotes how much time must happen between consecutive peaks in order for them to be classified as separate events. The peak window is the amount of time that the chlorophyll must remain above the threshold to be considered a bloom and the amount of time that the chlorophyll must be below the threshold for a termination date to occur. Fourteen days was chosen as the default event distance. Anything between 2-3 weeks gave the same information, so we chose to be more conservative with this value because of all the other parameters. The peak window is 10 day be default, which eliminates quick fluctuations above the threshold from the bloom event list. 
* Must include:
    * Dataset
    * Index (for climatology)
    * Event distance (event_distance = 14 by default)
    * Peak window (peak_window = 10 by default)